In [1]:
from huggingface_hub import InferenceClient
import json ,re
from prompts.task_graph import task_graph as task_graph_system_prompt
from prompts.task_graph_check import check_prompt as check_graph_system_prompt
import os

secret_value_0 = os.getenv("hf_token")

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# prompt = "What are the benefits of drinking green tea for weight loss?"
# prompt = "How many calories should I eat per day to maintain my current weight?"
# prompt = "My knees hurt when I run - should I stop running or just reduce distance?"
# prompt = "What's the difference between whey protein and plant-based protein?"# prompt = "Research the best exercises for lower back pain, then create a 4-week workout plan, and finally suggest dietary changes to support recovery."
# prompt = "Compare the benefits of HIIT vs steady-state cardio for fat loss, and recommend which one is better for someone with high blood pressure."
# prompt = "Analyze my sleep schedule (I sleep 5-6 hours), suggest improvements, and explain how poor sleep affects muscle recovery."
# prompt = "I want to lose 15 pounds in 3 months. First, calculate my daily calorie deficit needed. Then design a meal plan with macros. Finally, create a workout routine that fits with a busy work schedule."
# prompt = "Search for the latest research on intermittent fasting for diabetes management, summarize the key findings, and then explain whether it's safe for someone taking metformin."
prompt = "I'm a beginner wanting to start strength training. Explain proper form for the big 3 lifts (squat, bench, deadlift), create a beginner program, and list common mistakes to avoid. After that, suggest supplements that might help with muscle growth."

In [3]:
def extract_json(response):
    """Extract JSON from LLM response that may contain extra text"""
    try:
        # Try direct parsing first
        return json.loads(response)
    except json.JSONDecodeError:
        # Extract JSON from markdown code blocks or mixed text
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            return json.loads(json_str)
        raise ValueError("No valid JSON found in response")

In [4]:
task_graph_prompt =  f"User Prompt: {prompt} \n \nGenerate the dependency graph."

In [ ]:
def agent(prompt , system_prompt):
    client = InferenceClient(api_key=secret_value_0)

    completion = client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[
            system_prompt,
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    
    response = completion.choices[0].message.content.strip()
    return response
    

In [ ]:
response_graph = agent(task_graph_prompt , task_graph_system_prompt)
task_graph = extract_json(response_graph)['tasks']

In [7]:
task_graph

[{'id': 't1',
  'description': 'Explain proper form for the big 3 lifts (squat, bench, deadlift)',
  'depends_on': [],
  'agent': 'Exercise'},
 {'id': 't2',
  'description': 'Create a beginner program',
  'depends_on': ['t1'],
  'agent': 'Exercise'},
 {'id': 't3',
  'description': 'List common mistakes to avoid',
  'depends_on': ['t1', 't2'],
  'agent': 'Exercise'},
 {'id': 't4',
  'description': 'Suggest supplements that might help with muscle growth',
  'depends_on': ['t1', 't2', 't3'],
  'agent': 'Diet'}]

In [8]:
task_graph_updation_prompt = f"User Prompt: {prompt} Task: {task_graph} Return the updated task."

In [ ]:
response_updated_graph = agent(task_graph_updation_prompt , check_graph_system_prompt)
updated_graph = extract_json(response_updated_graph)['tasks']

In [ ]:
updated_graph

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


[{'id': 't1',
  'description': 'Explain proper form for the big 3 lifts (squat, bench, deadlift)',
  'depends_on': [],
  'agent': 'Exercise'},
 {'id': 't2',
  'description': 'Create a beginner program',
  'depends_on': ['t1'],
  'agent': 'Exercise'},
 {'id': 't3',
  'description': 'List common mistakes to avoid',
  'depends_on': ['t1', 't2'],
  'agent': 'Exercise'},
 {'id': 't4',
  'description': 'Suggest supplements that might help with muscle growth',
  'depends_on': ['t1', 't2', 't3'],
  'agent': 'Diet'}]

In [ ]:
from utils.central_database import save_data
save_data(updated_graph)

✓ Database created successfully at: database/central.db
✓ Inserted 4 tasks

Verification - Data in database:
  t1: Explain proper form for the big 3 lifts (squat, be... (depends on: [], agent: Exercise, resolved: 0)
  t2: Create a beginner program... (depends on: ["t1"], agent: Exercise, resolved: 0)
  t3: List common mistakes to avoid... (depends on: ["t1", "t2"], agent: Exercise, resolved: 0)
  t4: Suggest supplements that might help with muscle gr... (depends on: ["t1", "t2", "t3"], agent: Diet, resolved: 0)
